# 🩺 BioEcho v2 — Notebook 5: Face Disease Biomarkers
**Detects:** Anaemia (conjunctival pallor) · Jaundice (scleral yellowing) · Cardiovascular risk · Skin conditions

| Feature | v2 |
|---|---|
| Backbone | EfficientNet-V2-S |
| Optimizer | AdamW-8bit |
| Precision | BF16 |
| Compile | torch.compile reduce-overhead |
| Augmentation | Mixup + CutMix + colour jitter |
| Loss | Gaussian NLL + Kendall-Gal |
| Calibration | ECE + reliability diagrams |
| CV | K-Fold (3 folds) |
| Export | FP32 → INT8 ONNX |
| Autosave | Full state every epoch |

**Kaggle:** saroopmandal / KGAT_2a969618d36d94f56d0989908ec94774

In [ ]:
import json; from pathlib import Path
kd = Path.home()/'.kaggle'; kd.mkdir(exist_ok=True)
cp = kd/'kaggle.json'
# On Kaggle, credentials are auto-injected. No hardcoded key needed.
import os
json.dump({'username': os.environ.get('KAGGLE_USERNAME', 'saroopmandal'),
           'key': os.environ.get('KAGGLE_KEY', '')}, open(cp, 'w'))
cp.chmod(0o600); print('✅ Kaggle creds set')

In [ ]:
import subprocess
def run(cmd, timeout=900):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0 and r.stderr: print(f'[{cmd[:40]}]: {r.stderr[:200]}')
    return r
run('pip install -q --upgrade pip')
run('pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121', timeout=600)
run('pip install -q bitsandbytes>=0.43.0')
run('pip install -q opencv-python-headless albumentations scipy scikit-learn einops rich matplotlib seaborn pandas torchmetrics')
run('pip install -q onnx onnxruntime-gpu')
from onnxruntime.quantization import quantize_dynamic, QuantType
print('✅ All installed')

In [ ]:
import os, gc, time, warnings, random, math, urllib.request, zipfile
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.calibration import calibration_curve
from rich.console import Console; from rich.table import Table
import albumentations as A; from albumentations.pytorch import ToTensorV2
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
# GradScaler: use torch.amp.GradScaler (torch.cuda.amp is deprecated)
import bitsandbytes as bnb
import onnx, onnxruntime as ort

warnings.filterwarnings('ignore'); console = Console()
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE    = torch.float16
console.print(f'[bold green]GPUs:{NUM_GPUS} | BF16 | {DEVICE}[/]')

In [ ]:
@dataclass
class FaceConfig:
    data_dir:  str = '/kaggle/working/bioecho/face/data'
    ckpt_dir:  str = '/kaggle/working/bioecho/face/checkpoints'
    out_dir:   str = '/kaggle/working/bioecho/face/outputs'
    img_size:  int = 224
    embed_dim: int = 256
    dropout:   float = 0.3
    batch_size: int = 64
    grad_accum: int = 2
    epochs:    int = 12
    lr:        float = 3e-4
    backbone_lr: float = 5e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    patience:  int = 4
    ema_decay: float = 0.9995
    n_folds:   int = 2

C = FaceConfig()
for d in [C.data_dir, C.ckpt_dir, C.out_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

BINARY_TASKS  = ['anaemia', 'jaundice', 'cardiovascular_risk', 'skin_condition']
REGRESS_TASKS = ['pallor_score', 'redness_score']
ALL_TASKS     = BINARY_TASKS + REGRESS_TASKS
console.print(f'[cyan]EfficientNet-V2-S | Tasks: {ALL_TASKS}[/]')

In [ ]:
CKPT_DIR = Path(C.ckpt_dir)

def save_ckpt(epoch, fold, ms, ema_s, opt, sch, scl, hist, best):
    p = CKPT_DIR / f'face_ep{epoch:03d}_fold{fold}.pt'
    torch.save({'epoch':epoch,'fold':fold,'model_state':ms,'ema_shadow':ema_s,
                'opt':opt,'sch':sch,'scl':scl,'hist':hist,'best':best}, p)
    old = sorted(CKPT_DIR.glob(f'face_ep*_fold{fold}.pt'),
                 key=lambda x: int(x.stem.split('ep')[1].split('_')[0]))[:-2]
    for o in old: o.unlink(missing_ok=True)
    return p

def find_latest(fold=0):
    ckpts = sorted(CKPT_DIR.glob(f'face_ep*_fold{fold}.pt'),
                   key=lambda x: int(x.stem.split('ep')[1].split('_')[0]))
    return ckpts[-1] if ckpts else None

latest = find_latest(fold=0)
RESUME = str(latest) if latest else ''
console.print(f'[{"yellow" if latest else "green"}]{"▶ Resume: " + latest.name if latest else "✅ Fresh run"}[/]')

In [ ]:
DATA = Path(C.data_dir)

# ── HAM10000 skin lesions (Kaggle, free)
ham_ok = False
try:
    r = subprocess.run(
        f'kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection -p {DATA}/ham --unzip',
        shell=True, capture_output=True, text=True, timeout=600
    )
    ham_ok = r.returncode == 0
    console.print('[green]✅ HAM10000 loaded[/]' if ham_ok else f'[yellow]HAM: {r.stderr[:100]}[/]')
except Exception as e:
    console.print(f'[yellow]HAM skip: {e}[/]')

# ── Kaggle Anaemia dataset
anaemia_ok = False
try:
    r = subprocess.run(
        f'kaggle datasets download -d longnguyen2306/anemia-detection-from-conjunctiva-images -p {DATA}/anaemia --unzip',
        shell=True, capture_output=True, text=True, timeout=300
    )
    anaemia_ok = r.returncode == 0
    console.print('[green]✅ Anaemia dataset loaded[/]' if anaemia_ok else f'[yellow]Anaemia: {r.stderr[:100]}[/]')
except Exception as e:
    console.print(f'[yellow]Anaemia skip: {e}[/]')

# ── CelebA (faces — used for pallor/redness colour signal)
celeba_ok = False
try:
    r = subprocess.run(
        f'kaggle datasets download -d jessicali9530/celeba-dataset -p {DATA}/celeba --unzip',
        shell=True, capture_output=True, text=True, timeout=600
    )
    celeba_ok = r.returncode == 0
    console.print('[green]✅ CelebA loaded[/]' if celeba_ok else f'[yellow]CelebA: {r.stderr[:100]}[/]')
except Exception as e:
    console.print(f'[yellow]CelebA skip: {e}[/]')

console.print('[bold]\n✅ Dataset stage done[/]')

In [ ]:
# ── Synthetic face disease image generator
# Generates realistic synthetic faces with controlled colour biomarkers:
# - Pallor (anaemia): reduced redness, paler skin
# - Jaundice: yellowish sclera and skin
# - Cardiovascular: flushed redness, periorbital darkness

synth_dir = DATA / 'synthetic_faces'; synth_dir.mkdir(exist_ok=True)

def gen_face_image(anaemia=0.0, jaundice=0.0, cardio=0.0, skin_cond=0.0, noise=0.05):
    img = np.ones((C.img_size, C.img_size, 3), dtype=np.float32)
    # Base skin tone
    skin_r = 0.75 - anaemia*0.35 + cardio*0.15
    skin_g = 0.55 - anaemia*0.20 + jaundice*0.20
    skin_b = 0.45 - anaemia*0.15 - jaundice*0.10
    skin = np.array([skin_r, skin_g, skin_b]).clip(0,1)

    img[:,:] = skin

    # Face oval (vectorized)
    cx, cy = C.img_size//2, C.img_size//2
    yy, xx = np.mgrid[0:C.img_size, 0:C.img_size]
    outside = ((xx-cx)/80)**2 + ((yy-cy)/100)**2 > 1.0
    img[outside] = [0.85, 0.85, 0.85]

    # Eyes (sclera — jaundice turns them yellow) — vectorized
    sclera = np.array([1.0, 1.0-jaundice*0.5, 1.0-jaundice*0.7]).clip(0,1)
    for ey,ex,er in [(cy-20, cx-35, 18), (cy-20, cx+35, 18)]:
        ey_y, ey_x = np.mgrid[max(0,ey-er):min(C.img_size,ey+er),
                              max(0,ex-er):min(C.img_size,ex+er)]
        mask = (ey_x-ex)**2 + (ey_y-ey)**2 < er**2
        img[ey_y[mask], ey_x[mask]] = sclera

    # Conjunctiva region (inner corner — pallor = pale pink/white)
    conj = np.array([0.9-anaemia*0.5, 0.5-anaemia*0.3, 0.5-anaemia*0.3]).clip(0,1)
    for ey,ex in [(cy-20,cx-50),(cy-20,cx+50)]:
        img[ey-8:ey+8, ex-8:ex+8] = conj

    # Skin condition patches — vectorized
    if skin_cond > 0.3:
        for _ in range(int(skin_cond*8)):
            px = random.randint(cx-60,cx+60); py = random.randint(cy-60,cy+60)
            r = random.randint(3,12)
            sy, sx = np.mgrid[max(0,py-r):min(C.img_size,py+r),
                              max(0,px-r):min(C.img_size,px+r)]
            smask = (sx-px)**2 + (sy-py)**2 < r**2
            img[sy[smask], sx[smask]] = [0.7, 0.3, 0.3]

    # Periorbital darkness (cardiovascular)
    if cardio > 0.3:
        for ey,ex in [(cy-20,cx-35),(cy-20,cx+35)]:
            d = cardio * 0.4
            img[ey-25:ey-5, ex-25:ex+25] = np.clip(img[ey-25:ey-5, ex-25:ex+25]-d, 0, 1)

    # Gaussian noise
    img = np.clip(img + np.random.randn(*img.shape).astype(np.float32)*noise, 0, 1)
    return (img*255).astype(np.uint8)

N_SYNTH = 5000
synth_records = []
for i in range(N_SYNTH):
    an = float(np.random.beta(0.5,2))
    jd = float(np.random.beta(0.3,3))
    cv = float(np.random.beta(0.5,2))
    sk = float(np.random.beta(0.4,2))
    pl = an*0.8 + (1-cv)*0.2
    rd = cv*0.7 + sk*0.3
    img = gen_face_image(an, jd, cv, sk)
    p   = synth_dir / f'face_{i:05d}.jpg'
    cv2.imwrite(str(p), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    synth_records.append({'path':str(p),'anaemia':int(an>0.4),'jaundice':int(jd>0.3),
                          'cardiovascular_risk':int(cv>0.4),'skin_condition':int(sk>0.4),
                          'pallor_score':pl,'redness_score':rd,'source':'synth'})
    if i%1000==0: console.print(f'  Synth {i}/{N_SYNTH}')

console.print(f'[green]✅ {N_SYNTH} synthetic faces[/]')

In [ ]:
# ── Build full record list from all available sources
all_records = synth_records.copy()

# HAM10000 skin lesions → skin_condition=1, others=0
if ham_ok and (DATA/'ham').exists():
    for img_dir in [(DATA/'ham'/'HAM10000_images_part_1'),(DATA/'ham'/'HAM10000_images_part_2')]:
        if not img_dir.exists(): continue
        imgs = list(img_dir.glob('*.jpg'))[:2000]
        console.print(f'[cyan]HAM10000: {len(imgs)} images[/]')
        for p in imgs:
            all_records.append({'path':str(p),'anaemia':0,'jaundice':0,
                                'cardiovascular_risk':0,'skin_condition':1,
                                'pallor_score':0.2,'redness_score':0.6,'source':'ham'})

# Anaemia conjunctiva images
if anaemia_ok and (DATA/'anaemia').exists():
    for pth in (DATA/'anaemia').rglob('*.jpg'):
        is_an = 'positive' in str(pth).lower() or 'anaemia' in str(pth).lower()
        all_records.append({'path':str(pth),'anaemia':int(is_an),'jaundice':0,
                            'cardiovascular_risk':0,'skin_condition':0,
                            'pallor_score':0.7 if is_an else 0.1,'redness_score':0.1,'source':'anaemia'})

# CelebA — healthy faces (all labels=0)
if celeba_ok:
    for img_dir in [(DATA/'celeba'/'img_align_celeba'),(DATA/'celeba'/'img_celeba')]:
        if not img_dir.exists(): continue
        imgs = sorted(img_dir.glob('*.jpg'))[:2000]
        for p in imgs:
            all_records.append({'path':str(p),'anaemia':0,'jaundice':0,
                                'cardiovascular_risk':0,'skin_condition':0,
                                'pallor_score':0.1,'redness_score':0.2,'source':'celeba'})
        break

# Filter existing files
all_records = [r for r in all_records if Path(r['path']).exists()]
random.shuffle(all_records)
df_all = pd.DataFrame(all_records)
console.print(f'[bold]Total: {len(df_all)} | Anaemia:{df_all.anaemia.sum()} | Jaundice:{df_all.jaundice.sum()} | Skin:{df_all.skin_condition.sum()}[/]')

In [ ]:
# ── Augmentations: Albumentations (better colour handling than torchvision)
train_tfm = A.Compose([
    A.Resize(C.img_size, C.img_size),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.6),
    A.GaussianBlur(blur_limit=(3,7), p=0.2),
    # GaussNoise: var_limit renamed to noise_scale_factor in albumentations >= 1.4
    # Use a safe wrapper that works across versions
    A.OneOf([
        A.GaussNoise(p=1.0),   # albumentations >= 1.4 (uses default params)
    ], p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ToTensorV2(),
])
val_tfm = A.Compose([
    A.Resize(C.img_size, C.img_size),
    A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ToTensorV2(),
])

def mixup_imgs(x1, x2, y1, y2, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    return lam*x1 + (1-lam)*x2, {k: lam*y1[k]+(1-lam)*y2[k] for k in y1}

def cutmix_img(x, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    H, W = x.shape[1], x.shape[2]
    rw = int(W*np.sqrt(1-lam)); rh = int(H*np.sqrt(1-lam))
    cx = np.random.randint(W); cy = np.random.randint(H)
    x1,y1 = max(cx-rw//2,0), max(cy-rh//2,0)
    x2,y2 = min(cx+rw//2,W), min(cy+rh//2,H)
    x[:,y1:y2,x1:x2] = 0.0
    return x

class FaceDataset(Dataset):
    def __init__(self, df, tfm, is_train=True):
        self.df = df.reset_index(drop=True); self.tfm = tfm; self.is_train = is_train

    def __len__(self): return len(self.df)

    def _load(self, path):
        img = cv2.imread(path)
        if img is None:
            return np.zeros((C.img_size, C.img_size, 3), np.uint8)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = self._load(row['path'])
        img = self.tfm(image=img)['image']

        if self.is_train and random.random() < 0.2:
            img = cutmix_img(img)

        labels = {
            'anaemia':             torch.tensor(float(row['anaemia']),            dtype=torch.float32),
            'jaundice':            torch.tensor(float(row['jaundice']),           dtype=torch.float32),
            'cardiovascular_risk': torch.tensor(float(row['cardiovascular_risk']),dtype=torch.float32),
            'skin_condition':      torch.tensor(float(row['skin_condition']),     dtype=torch.float32),
            'pallor_score':        torch.tensor(float(row['pallor_score']),       dtype=torch.float32),
            'redness_score':       torch.tensor(float(row['redness_score']),      dtype=torch.float32),
        }
        return {'img': img, 'labels': labels}

console.print('[green]✅ Dataset classes ready[/]')

In [ ]:
# ── EfficientNet-V2-S multi-task face disease model

class FaceDiseaseModelV2(nn.Module):
    """
    EfficientNet-V2-S backbone + multi-task heads.
    Gaussian NLL outputs + Kendall-Gal learned task weighting.
    Gradient checkpointing on backbone.
    """
    def __init__(self, cfg: FaceConfig):
        super().__init__()
        # Load pretrained
        backbone = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        # Freeze first half of features
        feats = list(backbone.features.children())
        self.frozen_feats   = nn.Sequential(*feats[:4])
        self.trainable_feats = nn.Sequential(*feats[4:])
        for p in self.frozen_feats.parameters(): p.requires_grad_(False)

        self.pool = nn.AdaptiveAvgPool2d(1)
        # EfficientNet-V2-S last channel = 1280
        self.embed_proj = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(1280, 512), nn.GELU(),
            nn.Linear(512, cfg.embed_dim), nn.LayerNorm(cfg.embed_dim)
        )

        # Kendall-Gal learnable task log-variances
        self.log_vars = nn.Parameter(torch.zeros(len(ALL_TASKS)))

        # Task heads: binary → single logit, regress → (mean, log_var)
        self.heads = nn.ModuleDict()
        for t in BINARY_TASKS:
            self.heads[t] = nn.Sequential(
                nn.Linear(cfg.embed_dim, 64), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(64, 1)
            )
        for t in REGRESS_TASKS:
            self.heads[t] = nn.Sequential(
                nn.Linear(cfg.embed_dim, 64), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(64, 2)
            )

    def _backbone_forward(self, x):
        x = self.frozen_feats(x)
        return self.trainable_feats(x)

    def forward(self, img):
        # Gradient checkpointing on trainable features
        if self.training:
            import torch.utils.checkpoint as gc
            feats = gc.checkpoint(self._backbone_forward, img, use_reentrant=False)
        else:
            feats = self._backbone_forward(img)

        embed = self.embed_proj(self.pool(feats).flatten(1))  # (B, embed_dim)
        preds = {t: self.heads[t](embed) for t in self.heads}
        return embed, preds, self.log_vars


def build_model():
    m = FaceDiseaseModelV2(C).to(DEVICE)
    if NUM_GPUS > 1: m = nn.DataParallel(m)
    try:
        m = torch.compile(m, mode='reduce-overhead')
        console.print('[green]✅ torch.compile[/]')
    except: pass
    return m

def get_raw(m):
    r = m.module if hasattr(m, 'module') else m
    try: return r._orig_mod
    except: return r

# Sanity check
test_m = build_model()
with torch.no_grad():
    d = torch.randn(2, 3, C.img_size, C.img_size).to(DEVICE)
    emb, preds, lvs = get_raw(test_m)(d)
    console.print(f'Embed:{emb.shape} | {list(preds.keys())}')
del test_m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
bce = nn.BCEWithLogitsLoss()

def gnll(pred, tgt):
    mu, lv = pred[:,0], pred[:,1]
    var = lv.exp().clamp(1e-6)
    return (0.5*(lv + (mu-tgt)**2/var)).mean()

def kg(loss, lv):
    return (torch.exp(-lv)*loss + lv).mean()

def compute_loss(preds, labels, log_vars):
    losses = []
    for i, t in enumerate(BINARY_TASKS):
        losses.append(kg(bce(preds[t].squeeze(-1), labels[t]), log_vars[i]))
    for j, t in enumerate(REGRESS_TASKS):
        losses.append(kg(gnll(preds[t], labels[t]), log_vars[len(BINARY_TASKS)+j]))
    return sum(losses)

In [ ]:
class EMA:
    def __init__(self, m, d=0.9995):
        self.d = d; r = get_raw(m)
        self.shadow = {n:p.data.clone().cpu() for n,p in r.named_parameters() if p.requires_grad}
    def update(self, m):
        r = get_raw(m)
        for n,p in r.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.d*self.shadow[n] + (1-self.d)*p.data.cpu()

# ── K-Fold CV training
kf  = KFold(n_splits=C.n_folds, shuffle=True, random_state=SEED)
idx = np.arange(len(df_all))
all_fold_history = []

for fold, (tr_idx, vl_idx) in enumerate(kf.split(idx)):
    console.print(f'\n[bold cyan]═══ FOLD {fold+1}/{C.n_folds} ═══[/]')

    df_tr = df_all.iloc[tr_idx]; df_vl = df_all.iloc[vl_idx]
    tr_ds = FaceDataset(df_tr, train_tfm, is_train=True)
    vl_ds = FaceDataset(df_vl, val_tfm,   is_train=False)

    # Weighted sampler for anaemia (rarest condition)
    an_labels = df_tr['anaemia'].values.astype(int)
    cls_w = 1.0/(np.bincount(an_labels)+1e-8)
    sampler = WeightedRandomSampler(cls_w[an_labels], len(an_labels))
    tr_dl = DataLoader(tr_ds, batch_size=C.batch_size, sampler=sampler,  num_workers=4, pin_memory=True)
    vl_dl = DataLoader(vl_ds, batch_size=C.batch_size, shuffle=False,    num_workers=4, pin_memory=True)

    model = build_model()
    ema   = EMA(model, C.ema_decay)

    backbone_params  = list(get_raw(model).trainable_feats.parameters())
    other_params     = [p for n,p in get_raw(model).named_parameters()
                        if not n.startswith('trainable_feats') and not n.startswith('frozen_feats')
                        and p.requires_grad]
    try:
        opt = bnb.optim.AdamW8bit(
            [{'params':backbone_params,'lr':C.backbone_lr},
             {'params':other_params,   'lr':C.lr}],
            weight_decay=C.weight_decay
        )
        console.print('[green]✅ AdamW-8bit[/]')
    except:
        opt = torch.optim.AdamW(
            [{'params':backbone_params,'lr':C.backbone_lr},
             {'params':other_params,   'lr':C.lr}],
            weight_decay=C.weight_decay
        )

    total_steps = C.epochs * len(tr_dl) // C.grad_accum
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[C.backbone_lr, C.lr], total_steps=total_steps, pct_start=0.1
    )
    scl = torch.amp.GradScaler("cuda")

    # Resume
    latest_fold = find_latest(fold=fold)
    start_ep = 1; best_val = float('inf'); history = []; patience_cnt = 0
    if latest_fold:
        try:
            ck = torch.load(latest_fold, map_location=DEVICE)
            get_raw(model).load_state_dict(ck['model_state'])
            opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
            scl.load_state_dict(ck['scl']); ema.shadow = ck['ema_shadow']
            history = ck['hist']; best_val = ck['best']; start_ep = ck['epoch']+1
            console.print(f'[green]✅ Fold {fold} resumed ep{ck["epoch"]}[/]')
        except Exception as e:
            console.print(f'[red]Resume fail: {e}[/]'); start_ep = 1

    for epoch in range(start_ep, C.epochs+1):
        t0 = time.time(); model.train(); tl = 0.0
        ap = {t:[] for t in BINARY_TASKS}; ay = {t:[] for t in BINARY_TASKS}
        opt.zero_grad()

        for step, batch in enumerate(tr_dl):
            img    = batch['img'].to(DEVICE)
            labels = {k: v.to(DEVICE) for k,v in batch['labels'].items()}

            # Mixup augmentation
            if random.random() < 0.2:
                perm = torch.randperm(img.size(0))
                lam  = float(np.random.beta(0.4, 0.4))
                img  = lam*img + (1-lam)*img[perm]
                labels = {k: lam*v + (1-lam)*v[perm] for k,v in labels.items()}

            with torch.autocast('cuda', dtype=DTYPE):
                emb, preds, lvs = model(img)
                loss = compute_loss(preds, labels, lvs) / C.grad_accum

            scl.scale(loss).backward()
            if (step+1) % C.grad_accum == 0:
                scl.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), C.grad_clip)
                scl.step(opt); scl.update(); opt.zero_grad(); sch.step(); ema.update(model)

            tl += loss.item() * C.grad_accum
            for t in BINARY_TASKS:
                ap[t].extend(torch.sigmoid(preds[t].squeeze(-1)).detach().cpu().tolist())
                ay[t].extend(labels[t].detach().cpu().tolist())

        model.eval(); vl = 0.0
        vp = {t:[] for t in BINARY_TASKS}; vy = {t:[] for t in BINARY_TASKS}
        with torch.no_grad():
            for batch in vl_dl:
                img    = batch['img'].to(DEVICE)
                labels = {k: v.to(DEVICE) for k,v in batch['labels'].items()}
                with torch.autocast('cuda', dtype=DTYPE):
                    emb, preds, lvs = model(img)
                    vl += compute_loss(preds, labels, lvs).item()
                for t in BINARY_TASKS:
                    vp[t].extend(torch.sigmoid(preds[t].squeeze(-1)).cpu().tolist())
                    vy[t].extend(labels[t].cpu().tolist())

        vl /= len(vl_dl)
        aucs = {t: roc_auc_score(vy[t],vp[t]) if len(set(vy[t]))>1 else 0.5 for t in BINARY_TASKS}
        history.append({'epoch':epoch,'train_loss':tl/len(tr_dl),'val_loss':vl,
                        **{f'auc_{t}':v for t,v in aucs.items()}})
        console.print(
            f'F{fold+1} Ep{epoch:02d}/{C.epochs} | vl={vl:.4f} | '
            f'AN={aucs["anaemia"]:.3f} JD={aucs["jaundice"]:.3f} '
            f'CV={aucs["cardiovascular_risk"]:.3f} | {time.time()-t0:.0f}s'
        )

        save_ckpt(epoch, fold, get_raw(model).state_dict(), ema.shadow,
                  opt.state_dict(), sch.state_dict(), scl.state_dict(), history, vl)

        if vl < best_val:
            best_val = vl; patience_cnt = 0
            torch.save({'model_state':get_raw(model).state_dict(),'ema_shadow':ema.shadow,
                        'aucs':aucs,'fold':fold},
                       CKPT_DIR/f'face_best_fold{fold}.pt')
            console.print(f'  [green]✅ Best fold {fold}[/]')
        else:
            patience_cnt += 1
            if patience_cnt >= C.patience:
                console.print(f'[yellow]Early stop fold {fold}[/]'); break

    all_fold_history.append(history)
    del model, opt, sch; gc.collect(); torch.cuda.empty_cache()

json.dump(all_fold_history, open(Path(C.out_dir)/'face_history.json','w'))
console.print('[bold green]\n✅ K-Fold training done![/]')

In [ ]:
# ── Load best fold, calibration, ONNX export
best_fold = 0
best_path = CKPT_DIR/f'face_best_fold{best_fold}.pt'

final_model = FaceDiseaseModelV2(C).float().cpu().eval()
ck = torch.load(best_path, map_location='cpu')
final_model.load_state_dict(ck['model_state'])
for n,p in final_model.named_parameters():
    if p.requires_grad and n in ck['ema_shadow']:
        p.data.copy_(ck['ema_shadow'][n])

# Rebuild val loader for evaluation
tr_idx, vl_idx = list(KFold(n_splits=C.n_folds, shuffle=True, random_state=SEED).split(np.arange(len(df_all))))[best_fold]
vl_ds2 = FaceDataset(df_all.iloc[vl_idx], val_tfm, is_train=False)
vl_dl2 = DataLoader(vl_ds2, batch_size=C.batch_size, shuffle=False, num_workers=4)

vp3 = {t:[] for t in BINARY_TASKS}; vy3 = {t:[] for t in BINARY_TASKS}
final_model = final_model.to(DEVICE)
with torch.no_grad():
    for batch in vl_dl2:
        img = batch['img'].to(DEVICE)
        with torch.autocast('cuda', dtype=DTYPE):
            emb, preds, _ = final_model(img)
        for t in BINARY_TASKS:
            vp3[t].extend(torch.sigmoid(preds[t].squeeze(-1)).cpu().tolist())
            vy3[t].extend(batch['labels'][t].tolist())

# ECE calibration plots
def ece(y,p,n=10):
    bins=np.linspace(0,1,n+1); e=0.0; N=len(y)
    for b in range(n):
        m=(p>=bins[b])&(p<bins[b+1])
        if m.sum()==0: continue
        e+=m.sum()/N*abs(y[m].mean()-p[m].mean())
    return e

fig, axes = plt.subplots(1, len(BINARY_TASKS), figsize=(5*len(BINARY_TASKS), 5))
for ax, t in zip(axes, BINARY_TASKS):
    y = np.array(vy3[t]); p = np.array(vp3[t])
    if len(np.unique(y)) < 2: ax.set_title(f'{t}\n(no pos)'); continue
    fp, mp = calibration_curve(y, p, n_bins=10)
    auc = roc_auc_score(y,p)
    ax.plot(mp, fp, 's-', label=f'ECE={ece(y,p):.3f}')
    ax.plot([0,1],[0,1],'k--', label='Perfect')
    ax.set_title(f'{t}\nAUC={auc:.3f}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('Face v2 — Reliability Diagrams')
plt.tight_layout()
plt.savefig(Path(C.out_dir)/'face_calibration.png', dpi=150); plt.show()

# Metrics table
tbl = Table(title='Face Disease v2 — Eval Metrics', show_lines=True)
for c in ['Task','AUC-ROC','AP','ECE']: tbl.add_column(c)
for t in BINARY_TASKS:
    y = np.array(vy3[t]); p = np.array(vp3[t])
    if len(np.unique(y)) < 2: continue
    tbl.add_row(t, f'{roc_auc_score(y,p):.3f}', f'{average_precision_score(y,p):.3f}', f'{ece(y,p):.3f}')
console.print(tbl)

# ONNX export
class FaceExport(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, img):
        emb, preds, _ = self.m(img)
        bin_s = torch.stack([preds[t].squeeze(-1) for t in BINARY_TASKS], -1)
        reg_s = torch.stack([preds[t][:,0] for t in REGRESS_TASKS], -1)
        return emb, bin_s, reg_s

exp_m = FaceExport(final_model.cpu().float()).eval()
dummy = torch.randn(1, 3, C.img_size, C.img_size)

fp32 = Path(C.out_dir)/'face_fp32.onnx'
torch.onnx.export(
    exp_m, dummy, fp32,
    input_names=['face_image'],
    output_names=['face_embedding','binary_risks','regression_scores'],
    dynamic_axes={k:{0:'B'} for k in ['face_image','face_embedding','binary_risks','regression_scores']},
    opset_version=17, do_constant_folding=True
)
onnx.checker.check_model(onnx.load(str(fp32)))

int8 = Path(C.out_dir)/'face_int8.onnx'
quantize_dynamic(str(fp32), str(int8), weight_type=QuantType.QInt8, optimize_model=True)

tbl2 = Table(title='Face ONNX Export', show_lines=True)
for c in ['Format','Size MB','RTX 3050 Ready']: tbl2.add_column(c)
tbl2.add_row('FP32', f'{fp32.stat().st_size/1e6:.1f}', '❌ too large')
tbl2.add_row('INT8', f'{int8.stat().st_size/1e6:.1f}', '✅ yes')
console.print(tbl2)

console.print('[bold green]\n✅ Face disease notebook complete![/]')
console.print(f'Embed: {C.embed_dim}-d → feeds fusion notebook')
console.print(f'INT8 ONNX: {int8}')